# ACTUNEO AI: set up a local model

This notebook chooses a language model that suits your machine, downloads it, and tries it out. After the download it works offline and costs nothing.

Run the cells one at a time. **Section 3 downloads a model**, which can be several gigabytes: read the summary it prints before you say yes.

A language model can be wrong about actuarial work. It is used here only to pick a function and fill in its arguments.

## 1. Install

Run once, then restart the kernel.

In [ ]:
# %pip install --upgrade "actuneo[ai]"

## 2. What can this machine run?

Nothing is downloaded here. Free memory is measured now, so close other applications first if you want a larger model.

In [ ]:
import actuneo.ai as ai

hardware = ai.detect_hardware()
print(hardware.describe())

selection = ai.select_model(hardware)
if selection.too_weak:
    print("No local model fits. Use ai.configure(provider=...) with your own key instead.")
else:
    for label, choice in [("Smallest", selection.smallest),
                          ("Recommended", selection.recommended),
                          ("Largest", selection.largest)]:
        print(f"{label:12s} {choice.model.key:8s} {choice.model.name:26s} "
              f"{choice.model.size_gb:5.1f} GB   {choice.runs_on()}")

## 3. Download and set up

**Best done in a terminal**, where you answer the question and watch the download's progress bar. Open a terminal in the same environment and run:

```
actuneo-ai models      # every model and whether it fits this machine
actuneo-ai setup       # shows the recommendation, asks, then downloads
```

or pick a model directly: `actuneo-ai setup --model tiny --force`. If the `actuneo-ai` command is not found, use `python -m actuneo.ai setup`.

If the download is interrupted, run the same command again: it resumes where it stopped. When it finishes, come back here and carry on from section 4.

You can also run it from the notebook with the cell below, but the progress bar may not show.

In [ ]:
# ai.setup(force=True)

To choose without being asked, use one of these instead (remove the `#`).

| Key | Model | Download |
|---|---|---|
| `minimal` | Qwen3 0.6B | 0.4 GB |
| `tiny` | Qwen3 1.7B | 1.1 GB |
| `small` | Qwen3 4B | 2.5 GB |
| `default` | Qwen3 8B | 5.0 GB |
| `strong` | Qwen3 14B | 9.0 GB |
| `max` | Qwen3 30B-A3B | 18.6 GB |

Challengers to compare: `phi4-mini`, `phi4-mini-reasoning`, `gemma3-4b`, `deepseek-r1-14b`.

In [ ]:
# ai.setup(auto=True, force=True)          # the recommended model, no questions
# ai.setup(model="tiny", force=True)        # a particular model
# ai.setup(model="largest", force=True)     # the most this machine can hold
# ai.setup(model="phi4-mini", force=True)   # a challenger

## 4. Check what is installed

In [ ]:
ai.status();

## 5. Try it

The first call loads the model, which takes a few seconds. Later calls are faster, and a repeated question is answered from memory.

In [ ]:
model = ai.get_backend()
print(model.generate("In one sentence, what is a pension?", max_tokens=80))

In [ ]:
tools = [
    {"name": "life_expectancy",
     "description": "Expectation of life, how long a person is expected to live, at an age",
     "parameters": {"type": "object",
                    "properties": {"age": {"type": "integer"},
                                   "sex": {"type": "string", "enum": ["M", "F"]}},
                    "required": ["age"]}},
    {"name": "annuity_value",
     "description": "Value of a pension of an annual amount paid for life from an age",
     "parameters": {"type": "object",
                    "properties": {"age": {"type": "integer"},
                                   "annual_amount": {"type": "number"},
                                   "interest_rate": {"type": "number",
                                                     "description": "as a decimal, 0.05 for 5%"}},
                    "required": ["age", "annual_amount", "interest_rate"]}},
]

for question in ["How long is a 40 year old woman expected to live?",
                 "Value a pension of 6000 a year from age 65 at 5% interest.",
                 "Good morning!"]:
    call = model.call_tools(question, tools)
    print(question)
    print("   ->", call.name, call.arguments, call.text[:60])

## 6. How good is this model on your machine?

Twenty actuarial requests over twelve tools. It reports how often the function is right, how often the function and its arguments are both right, and the seconds per request. Run it after each model you try, to compare them.

In [ ]:
results = ai.benchmark()
results.attrs["summary"]

In [ ]:
# The requests it got wrong
results.loc[~results["arguments_ok"], ["request", "expected", "chosen", "arguments"]]

## 7. Free the disk space

Deletes the model. It asks first.

In [ ]:
# ai.remove_model()

## 8. Or use your own API key instead

If the machine is too small for a local model. The key is kept in memory for this session and is never written to ACTUNEO's configuration file. Setting the environment variable (`GEMINI_API_KEY`, `OPENAI_API_KEY` or `ANTHROPIC_API_KEY`) works too.

In [ ]:
# ai.configure(provider="gemini", api_key="paste-your-key-here")   # or "openai", "anthropic", "ollama"
# ai.get_backend().generate("Hello")